In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

DATA_CSV = Path('sample_data/wdi_world_slice.csv')
TARGET = 'SP.DYN.CBRT.IN'
FEATURES = {
  'SP.DYN.TFRT.IN': 'fertility_rate',
  'NY.GDP.PCAP.CD': 'gdp_per_capita_usd',
  'SP.DYN.LE00.IN': 'life_expectancy',
  'SP.URB.TOTL.IN.ZS': 'urban_population_pct',
  'SE.XPD.TOTL.GD.ZS': 'education_expenditure_pct_gdp',
  'SL.UEM.TOTL.ZS': 'unemployment_pct',
  'SP.POP.GROW': 'population_growth_pct',
  'SH.DYN.MORT': 'under5_mortality_per1000',
}
CODES = [TARGET] + list(FEATURES.keys())

raw = pd.read_csv(DATA_CSV, low_memory=False)
years = [c for c in raw.columns if ' [YR' in c and c[:4].isdigit()]
long = raw.melt(id_vars=['Country Name', 'Country Code', 'Series Name', 'Series Code'], value_vars=years, var_name='year_col', value_name='value')
long['year'] = long['year_col'].str.slice(0, 4).astype(int)
long['value'] = pd.to_numeric(long['value'].replace('..', np.nan), errors='coerce')
long = long[long['Series Code'].isin(CODES)]
table = long.pivot_table(index='year', columns='Series Code', values='value', aggfunc='mean').reset_index()
table = table.rename(columns={TARGET: 'birth_rate', **FEATURES})
table = table.dropna(subset=['birth_rate']).sort_values('year')
sns.set_theme(style='whitegrid')

In [ ]:
table[['year', 'birth_rate', 'fertility_rate', 'gdp_per_capita_usd']]

In [ ]:
plt.plot(table['year'], table['birth_rate'], marker='o')
plt.show()

In [ ]:
sns.heatmap(table[['birth_rate', 'fertility_rate', 'gdp_per_capita_usd']].corr(), annot=True)
plt.show()